# 알약 탐지·분류 통합 추론

이 노트북은 학습된 탐지기와 분류기 `best.pt`를 불러와 원본 이미지에서 알약 위치와 세부 품목을 순서대로 예측하고 결과를 저장한다. 두 모델의 학습은 개별 학습 노트북에서 수행한다.

## 실행 환경

- **Colab:** 저장소, 두 checkpoint와 추론 이미지 경로를 준비한 뒤 실행한다.
- **로컬:** 저장소 루트에서 `uv sync --locked`를 실행하고 `data/inference/`에 추론 이미지를 준비한다. CUDA, MPS 또는 CPU 중 사용 가능한 장치를 자동으로 선택한다.

## 추론 실행 설정

설정 셀에서 탐지기·분류기 실행 이름, 입력 경로, batch 크기와 탐지 점수 임곗값을 지정한다. 결과는 `outputs/pill_recognition/`에 저장된다.

주요 결과물:

- `pill_predictions.json`: bbox, 품목 예측과 결합 점수
- `pill_recognition_errors.json`: 탐지 또는 분류 단계 오류
- `visualizations/`: 이미지별 최종 예측 시각화

In [ ]:
import sys
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from configs.pill_recognition import PillRecognitionConfig
from data_loading.factory import load_data_factory
from pipeline import plot_pill_predictions
from pipeline.pill_recognition import recognize_pills_from_config

In [ ]:
inference_run_name = "20261102-ssdlite320-resnet50-infer-mini-smoke-01"

# tags = [
#     "task:inference",
#     "stage:smoke",
#     "detector:ssdlite320",
#     "classifier:resnet50",
#     "data:infer-mini",
#     "device:mps",
# ]

output_root = project_root / "outputs"
detector_run_name = "detector_smoke_test_2"
classifier_run_name = "smoke_test2"
inference_factory_path = "data_loading.recognition:build_pill_recognition_inputs"
input_dir = project_root / "data" / "inference"
inference_batch_size = 1
score_threshold = 0.05

detector_run_dir = output_root / "detector" / detector_run_name
classifier_run_dir = output_root / "classifier" / classifier_run_name

detector_checkpoint_path = detector_run_dir / "best.pt"
classifier_checkpoint_path = classifier_run_dir / "best.pt"
detector_label_map_path = detector_run_dir / "label_map.json"
classifier_class_map_path = classifier_run_dir / "class_map.json"
result_dir = output_root / "inference" / inference_run_name

## 추론 데이터 연결

`input_dir`에 추론할 이미지를 넣으면 기본 factory가 재귀적으로 읽는다. 외부 factory를 사용할 경우 `factory(config=config, batch_size=inference_batch_size)` 호출을 지원하고 `images: list[Tensor]`와 `meta: list[dict]`를 반환해야 한다.

In [ ]:
config = PillRecognitionConfig(
    inference_factory_path=inference_factory_path,
    input_dir=input_dir,
    detector_checkpoint_path=detector_checkpoint_path,
    classifier_checkpoint_path=classifier_checkpoint_path,
    detector_label_map_path=detector_label_map_path,
    classifier_class_map_path=classifier_class_map_path,
    output_dir=result_dir,
    detector_name=detector_run_name,
    batch_size=inference_batch_size,
    score_threshold=score_threshold,
    show_progress=True,
    device="auto",
)
print("====================\npill recognition\n====================\n")
print(
    f"run_name: {inference_run_name}\n"
    f"device: {config.device}\n"
    f"input_dir: {config.input_dir}\n"
    f"result_dir: {config.output_dir}\n"
    f"detector_run: {detector_run_name}\n"
    f"classifier_run: {classifier_run_name}"
)
print(f"batch_size: {config.batch_size}")
print(f"score_threshold: {config.score_threshold}")

## Pill recognition 추론

두 checkpoint는 현재 label map 및 class map과 일치해야 한다. 모델은 같은 CUDA, MPS 또는 CPU 장치에서 순서대로 실행된다. 탐지 bbox를 이미지 경계로 보정해 crop하고 학습과 같은 padding·정규화를 적용한다. 최종 점수는 `detector_score × classifier_score`로 계산한다.

In [ ]:
result = recognize_pills_from_config(config)
print("통합 추론과 artifact 저장 완료")

## 추론 결과 확인

저장된 예측과 단계별 오류 수를 확인하고 표의 앞부분을 표시한다.

In [ ]:
predictions_path = result_dir / "pill_predictions.json"
errors_path = result_dir / "pill_recognition_errors.json"
print(
    f"predictions={len(result.predictions)}, "
    f"detector_errors={len(result.detector_errors)}, "
    f"classifier_errors={len(result.classifier_errors)}"
)
print(f"saved: {predictions_path}")
print(f"saved: {errors_path}")

In [ ]:
prediction_table = pd.DataFrame(
    [prediction.to_dict() for prediction in result.predictions]
)
display(prediction_table.head(20))

error_table = pd.DataFrame(
    [{"stage": "detector", **error.to_dict()} for error in result.detector_errors]
    + [{"stage": "classifier", **error.to_dict()} for error in result.classifier_errors]
)
display(error_table.head(20))

## 예측 시각화

첫 번째 추론 이미지에 최종 품목명, bbox와 결합 점수를 표시하고 PNG로 저장한다.

In [ ]:
inference_factory = load_data_factory(
    config.inference_factory_path,
    setting_name="inference_factory_path",
)
visualization_loader = inference_factory(
    config=config,
    batch_size=config.batch_size,
)
first_batch = next(iter(visualization_loader))
first_image = first_batch["images"][0]
first_image_id = str(first_batch["meta"][0]["image_id"])
first_predictions = [
    prediction
    for prediction in result.predictions
    if prediction.image_id == first_image_id
]
figure = plot_pill_predictions(
    first_image,
    first_predictions,
    title=f"Pill recognition image_id={first_image_id}",
)
visualization_dir = result_dir / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
figure.savefig(
    visualization_dir / f"image-{first_image_id}.png",
    bbox_inches="tight",
)
figure